# 01 — Pandas EDA (Gold/Dashboard Layer)

This notebook explores business behavior using curated warehouse tables (gold + dashboard), while still keeping a quick raw-level sanity view.

## Data sources used
- Gold: `gold_fact_orders`, `gold_fact_order_items`, `gold_fact_order_payments`, `gold_fact_order_reviews`, `gold_customer_delivered_orders`
- Dashboard marts: `dash_kpi_monthly`, `dash_kpi_daily`, `dash_retention_snapshot`, `dash_retention_monthly`, `dash_category_monthly`

## Sections
1. Business Understanding
2. Time Trend Exploration
3. Customer Behavior Exploration
4. Revenue Exploration
5. Category Exploration
6. Retention Signal Exploration

In [2]:
from pathlib import Path
import os
import importlib
import subprocess
import sys
import getpass
from urllib.parse import quote_plus


def ensure_pkg(module_name: str, pip_name: str) -> None:
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])


# Required DB dependencies for this notebook.
ensure_pkg("sqlalchemy", "sqlalchemy")
ensure_pkg("psycopg2", "psycopg2-binary")

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

ROOT = Path("..").resolve()

# Update these env vars if needed before running notebook.
DB_HOST = os.getenv("POSTGRES_HOST", "localhost")
DB_PORT = os.getenv("POSTGRES_PORT", "5432")
DB_NAME = os.getenv("POSTGRES_DB", "ecommerce")
DB_USER = os.getenv("POSTGRES_USER") or getpass.getuser()
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD", "")

password_part = quote_plus(DB_PASSWORD)
conn_str = f"postgresql+psycopg2://{DB_USER}:{password_part}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(conn_str)


def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(text(sql), engine)

try:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print(f"Connected target: {DB_HOST}:{DB_PORT}/{DB_NAME} as {DB_USER}")
except Exception as e:
    raise RuntimeError(
        "Database connection failed. Set env vars before running:\n"
        "POSTGRES_HOST, POSTGRES_PORT, POSTGRES_DB, POSTGRES_USER, POSTGRES_PASSWORD\n"
        f"Current tried user={DB_USER}, db={DB_NAME}, host={DB_HOST}:{DB_PORT}.\n"
        f"Original error: {e}"
    )

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 18.1 MB/s  0:00:00



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 27.7 MB/s  0:00:00



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


Connected target: localhost:5432/ecommerce


## 1) Business Understanding

In [3]:
orders = q("SELECT * FROM gold_fact_orders")
items = q("SELECT * FROM gold_fact_order_items")
payments = q("SELECT * FROM gold_fact_order_payments")
reviews = q("SELECT * FROM gold_fact_order_reviews")
cust_delivered = q("SELECT * FROM gold_customer_delivered_orders")

kpi_monthly = q("SELECT * FROM dash_kpi_monthly ORDER BY report_month")
kpi_daily = q("SELECT * FROM dash_kpi_daily ORDER BY report_date")
ret_snapshot = q("SELECT * FROM dash_retention_snapshot ORDER BY metric_code")
ret_monthly = q("SELECT * FROM dash_retention_monthly ORDER BY report_month, metric_code")
cat_monthly = q("SELECT * FROM dash_category_monthly ORDER BY report_month, category_rank")

shape_df = pd.DataFrame([
    ("gold_fact_orders", *orders.shape),
    ("gold_fact_order_items", *items.shape),
    ("gold_fact_order_payments", *payments.shape),
    ("gold_fact_order_reviews", *reviews.shape),
    ("gold_customer_delivered_orders", *cust_delivered.shape),
    ("dash_kpi_monthly", *kpi_monthly.shape),
    ("dash_retention_snapshot", *ret_snapshot.shape),
], columns=["table", "rows", "cols"])
shape_df

OperationalError: (psycopg2.OperationalError) connection to server at "localhost" (::1), port 5432 failed: FATAL:  role "postgres" does not exist

(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [ ]:
orders[["order_status", "is_delivered", "order_purchase_timestamp"]].describe(include="all")

In [ ]:
status_dist = (
    orders["order_status"].value_counts(dropna=False)
    .rename_axis("order_status")
    .reset_index(name="orders")
)
status_dist["pct"] = status_dist["orders"] / status_dist["orders"].sum()

purchase_range = {
    "min_purchase_ts": orders["order_purchase_timestamp"].min(),
    "max_purchase_ts": orders["order_purchase_timestamp"].max(),
}

critical_nulls = orders[[
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "order_payment_total",
    "order_avg_review_score",
]].isna().mean().rename("null_rate").to_frame()

status_dist, purchase_range, critical_nulls

## 2) Time Trend Exploration

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

sns.lineplot(data=kpi_monthly, x="report_month", y="total_orders", ax=axes[0, 0])
axes[0, 0].set_title("Monthly Delivered Orders")

sns.lineplot(data=kpi_monthly, x="report_month", y="gmv", label="GMV", ax=axes[0, 1])
sns.lineplot(data=kpi_monthly, x="report_month", y="actual_payment", label="Actual Payment", ax=axes[0, 1])
axes[0, 1].set_title("Monthly GMV vs Actual Payment")

sns.lineplot(data=kpi_monthly, x="report_month", y="aov", ax=axes[1, 0])
axes[1, 0].set_title("Monthly AOV")

sns.lineplot(data=kpi_monthly, x="report_month", y="delivery_delay_rate", ax=axes[1, 1])
axes[1, 1].set_title("Monthly Delivery Delay Rate")

for ax in axes.flat:
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## 3) Customer Behavior Exploration

In [ ]:
cust_orders = (
    cust_delivered.groupby("customer_unique_id", as_index=False)
    .agg(delivered_orders=("order_id", "nunique"))
)

summary = pd.DataFrame({
    "metric": ["one_time_customers", "repeat_customers_gt1", "loyal_customers_ge5"],
    "value": [
        (cust_orders["delivered_orders"] == 1).sum(),
        (cust_orders["delivered_orders"] > 1).sum(),
        (cust_orders["delivered_orders"] >= 5).sum(),
    ],
})
summary["rate"] = summary["value"] / len(cust_orders)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.histplot(cust_orders["delivered_orders"], bins=30, ax=axes[0])
axes[0].set_title("Orders per Customer (customer_unique_id)")
axes[0].set_xlim(1, cust_orders["delivered_orders"].quantile(0.99))

sns.lineplot(data=kpi_monthly, x="report_month", y="installment_usage_rate", ax=axes[1])
axes[1].set_title("Monthly Installment Usage Rate")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

summary

## 4) Revenue Exploration

In [ ]:
order_revenue = orders.loc[orders["is_delivered"], ["order_id", "customer_unique_id", "order_gmv"]].copy()

customer_revenue = (
    order_revenue.groupby("customer_unique_id", as_index=False)
    .agg(gmv=("order_gmv", "sum"))
    .sort_values("gmv", ascending=False)
)
customer_revenue["cum_gmv_pct"] = customer_revenue["gmv"].cumsum() / customer_revenue["gmv"].sum()
customer_revenue["cum_customer_pct"] = (customer_revenue.index + 1) / len(customer_revenue)

top20_cut = customer_revenue.loc[customer_revenue["cum_customer_pct"] <= 0.2, "gmv"].sum() / customer_revenue["gmv"].sum()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.histplot(order_revenue["order_gmv"], bins=50, ax=axes[0])
axes[0].set_title("Order-level GMV Distribution")

axes[1].plot(customer_revenue["cum_customer_pct"], customer_revenue["cum_gmv_pct"])
axes[1].axvline(0.2, linestyle="--", color="red")
axes[1].set_title("Pareto Curve (Customer GMV Contribution)")
axes[1].set_xlabel("Cumulative share of customers")
axes[1].set_ylabel("Cumulative share of GMV")

plt.tight_layout()
plt.show()

print(f"Top 20% customers contribute {top20_cut:.2%} of GMV")

## 5) Category Exploration

In [ ]:
cat_top = (
    cat_monthly.groupby("category", as_index=False)
    .agg(total_gmv=("category_gmv", "sum"), total_orders=("category_order_count", "sum"))
)
cat_top["category_aov"] = cat_top["total_gmv"] / cat_top["total_orders"]
cat_top = cat_top.sort_values("total_gmv", ascending=False)

unknown_share = cat_top.loc[cat_top["category"] == "Unknown", "total_gmv"].sum() / cat_top["total_gmv"].sum()

plt.figure(figsize=(10, 5))
sns.barplot(data=cat_top.head(10), x="total_gmv", y="category")
plt.title("Top 10 Categories by GMV")
plt.tight_layout()
plt.show()

print(f"Unknown category GMV share: {unknown_share:.2%}")
cat_top.head(15)

## 6) Retention Signal Exploration

In [ ]:
ret_snapshot

In [ ]:
plt.figure(figsize=(12, 5))
plot_df = ret_monthly.copy()
plot_df["metric_code"] = pd.Categorical(
    plot_df["metric_code"],
    categories=[
        "lifetime_repeat_purchase",
        "last_3m_returning_customer",
        "last_6m_returning_customer",
        "last_3m_repeat_purchase",
        "last_6m_repeat_purchase",
    ],
    ordered=True,
)

sns.lineplot(
    data=plot_df.sort_values(["report_month", "metric_code"]),
    x="report_month",
    y="repeat_purchase_rate",
    hue="metric_code",
)
plt.title("Retention Metrics Trend (Monthly)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()